<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v3/data-loading/streaming-dataset/streaming-dataset.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("streaming-dataset", ...)`


# Streaming Dataset with IterableDataset

**Difficulty**: 🟡 Medium

---

### Problem Statement

When data is too large to fit in memory, or arrives as a stream (log files, network sockets, database cursors), you can't use a map-style dataset with random `__getitem__` access. PyTorch's `IterableDataset` solves this: you implement `__iter__` as a Python **generator** that `yield`s one sample at a time.

Your task is to implement a streaming dataset that reads a large file **line by line** without loading it all into memory.

---

### Requirements

1. **`StreamingDataset(filepath)`** — A `torch.utils.data.IterableDataset` subclass:
   - `__iter__` opens the file and yields `(x, y)` tensor pairs one line at a time.
   - Must **not** read the entire file into memory.

2. **Multi-worker correctness** (conceptual):
   - When `DataLoader` uses `num_workers > 1`, each worker gets its own copy of the iterator. Without handling this, every worker yields the **entire** file (duplicated data!).
   - Implement worker-aware splitting using `torch.utils.data.get_worker_info()` so each worker processes a disjoint shard.
   - *Note: real worker processes are awkward to test from a notebook: with the `spawn` start method (the default on macOS and Windows) a class defined in a notebook cell cannot be pickled into the workers. The tests below therefore fake the context a `DataLoader` worker runs in (`get_worker_info()` reporting worker `k` of `n`) and check that the shards are disjoint and together cover the file; on Linux they also run a real `DataLoader(num_workers=2)`.*

3. **Generate test data** — Create a CSV with 10,000 rows to test against.

---

### Constraints

- ✅ Memory usage must be O(1) per sample, not O(N).
- ✅ With `num_workers=0` (main process), all samples must be yielded.
- ✅ Your `__iter__` must use `get_worker_info()` to support multi-worker sharding.
- ❌ Do **not** use `pandas.read_csv()` or load the whole file at once.

---

<details>
  <summary>💡 Hint</summary>

  - Use `open(filepath)` and iterate line by line — Python file objects are lazy iterators.
  - `get_worker_info()` returns `None` in the main process, or a `WorkerInfo` with `.id` and `.num_workers`.
  - To shard: use `enumerate()` on lines, and each worker only yields lines where `line_idx % num_workers == worker_id`.
  - `DataLoader` does not accept `shuffle=True` for an `IterableDataset` (it raises a `ValueError`) — shuffling must be done differently (e.g., a shuffle buffer, covered in the next exercise).

</details>

---

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info
import os
import tempfile

# Generate synthetic CSV: 10,000 rows, 5 features + 1 target
torch.manual_seed(42)
N, D = 10_000, 5
X = torch.randn(N, D)
y = X @ torch.randn(D) + 0.1 * torch.randn(N)

data_path = os.path.join(tempfile.gettempdir(), "streaming_test_data.csv")
with open(data_path, "w") as f:
    for i in range(N):
        row = ",".join(f"{v:.6f}" for v in X[i].tolist()) + f",{y[i]:.6f}\n"
        f.write(row)

print(f"Wrote {N} rows to {data_path}")
print(f"File size: {os.path.getsize(data_path) / 1024:.1f} KB")

In [ ]:
# Peek at the file format: 5 comma-separated features, then the target
with open(data_path) as f:
    for _ in range(3):
        print(f.readline().rstrip())

In [ ]:
class StreamingDataset(IterableDataset):
    """Streams (x, y) pairs from a CSV file, one line at a time."""

    def __init__(self, filepath: str):
        super().__init__()
        self.filepath = filepath

    def __iter__(self):
        # TODO: Get worker info for multi-worker sharding
        # worker_info = get_worker_info()
        # If None → single process, yield everything
        # Otherwise → only yield lines where line_idx % num_workers == worker_id

        # TODO: Open file and iterate line by line
        # For each line: parse comma-separated floats, split into x (all but last) and y (last)
        # Yield (x_tensor, y_tensor)
        ...

In [ ]:
# === Tests ===

# Test 1: Single-process iteration yields all samples
ds = StreamingDataset(data_path)
loader = DataLoader(ds, batch_size=64)
total = sum(by.shape[0] for _, by in loader)
assert total == N, f"Expected {N} samples, got {total}"
print(f"PASSED: Single-process yields all {N} samples")

# Test 2: Correct shapes
ds2 = StreamingDataset(data_path)
loader2 = DataLoader(ds2, batch_size=32)
bx, by = next(iter(loader2))
assert bx.shape == (32, D), f"Expected X shape (32, {D}), got {bx.shape}"
assert by.shape == (32,), f"Expected y shape (32,), got {by.shape}"
print("PASSED: Correct batch shapes")

# Test 3: Values match original data
ds3 = StreamingDataset(data_path)
loader3 = DataLoader(ds3, batch_size=N)
bx_all, by_all = next(iter(loader3))
assert torch.allclose(bx_all, X, atol=1e-4), "X values don't match"
assert torch.allclose(by_all, y, atol=1e-4), "y values don't match"
print("PASSED: Streamed values match original tensors")

# Test 4: Worker sharding. Inside a DataLoader worker, get_worker_info() reports the worker's
# id; we fake that context here (a private hook) so the test runs on every OS.
from contextlib import contextmanager
from torch.utils.data._utils import worker as _worker

@contextmanager
def as_worker(worker_id, num_workers, dataset):
    _worker._worker_info = _worker.WorkerInfo(id=worker_id, num_workers=num_workers, seed=0, dataset=dataset)
    try:
        yield
    finally:
        _worker._worker_info = None

for num_workers in (2, 3):
    shards = []
    for worker_id in range(num_workers):
        ds4 = StreamingDataset(data_path)
        with as_worker(worker_id, num_workers, ds4):
            shards.append(torch.stack([sy for _, sy in ds4]))
    sizes = [len(s) for s in shards]
    assert sum(sizes) == N, \
        f"{num_workers} workers yielded {sum(sizes)} samples in total, expected {N} — rows are duplicated or lost"
    assert torch.allclose(torch.cat(shards).sort().values, y.sort().values, atol=1e-4), \
        "Together the shards must contain every row exactly once"
    assert min(sizes) > N / (2 * num_workers), f"Shards are very unbalanced: {sizes}"
print("PASSED: Each simulated worker gets a disjoint shard; together they cover the file")

# Test 5: The real thing - two worker processes (Linux only: needs the "fork" start method,
# which, unlike spawn, does not pickle the notebook-defined class)
import sys
if sys.platform.startswith("linux"):
    loader5 = DataLoader(StreamingDataset(data_path), batch_size=64, num_workers=2,
                         multiprocessing_context="fork")
    y5 = torch.cat([by for _, by in loader5])
    assert len(y5) == N, f"With 2 workers got {len(y5)} samples, expected {N} — are the workers duplicating rows?"
    assert torch.allclose(y5.sort().values, y.sort().values, atol=1e-4), "Multi-worker loading lost or duplicated rows"
    print("PASSED: DataLoader(num_workers=2) yields every row exactly once")
else:
    print("SKIPPED: real multi-worker test (needs Linux)")

print("\nAll tests passed!")